# Fraud Detection Pipeline — Hail Damage Claims

**Project:** Travelers Insurance x UConn AI/ML Research Fellowship  
**Author:** Anushree Sabade

This notebook implements a basic fraud detection pipeline for hail damage drone images.
It does two main things:

1. **Feature Extraction** — Uses a pretrained ResNet-18 to pull deep visual features from the image.
2. **Metadata Analysis** — Checks the image's EXIF data for GPS coordinates and a timestamp.
   Missing or suspicious metadata is flagged as a potential fraud signal.

In a real production system, the extracted features would feed into a classifier trained on
labeled fraudulent vs. legitimate claims. The metadata checks would combine with that score
for a final risk assessment.

## Step 1: Imports

We use:
- `torch` / `torchvision` for the pretrained ResNet model
- `PIL` (Pillow) for loading the image and reading EXIF data
- `numpy` for handling the feature vector
- `os` and `datetime` from the standard library

In [ ]:
import os
import datetime

import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ExifTags

## Step 2: Load a Sample Image

For this demo we create a small placeholder image (just random pixels) since we don't have
a real drone image here. In practice you'd swap this path for an actual `.jpg` from the drone.

The placeholder still lets us run the full pipeline end-to-end so you can see what the output looks like.

In [ ]:
SAMPLE_IMAGE_PATH = "sample_roof.jpg"

# Create a placeholder image if one doesn't already exist
if not os.path.exists(SAMPLE_IMAGE_PATH):
    print("No sample image found — creating a placeholder (random noise image).")
    placeholder = Image.fromarray(
        np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    )
    placeholder.save(SAMPLE_IMAGE_PATH)
    print(f"Placeholder saved to: {SAMPLE_IMAGE_PATH}")
else:
    print(f"Loaded image from: {SAMPLE_IMAGE_PATH}")

image = Image.open(SAMPLE_IMAGE_PATH).convert("RGB")
print(f"Image size: {image.size}, Mode: {image.mode}")

## Step 3: Preprocess the Image

ResNet expects images to be:
- Resized to 224×224 pixels
- Converted to a tensor (pixel values 0.0–1.0)
- Normalized using the ImageNet mean and standard deviation
  (because ResNet was trained on ImageNet, so we match that scale)

We also add a batch dimension with `unsqueeze(0)` — PyTorch models always
expect a batch of images, even when you only have one.

In [ ]:
# Standard ImageNet normalization values
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Apply the transforms and add batch dimension
input_tensor = preprocess(image).unsqueeze(0)  # shape: (1, 3, 224, 224)
print(f"Input tensor shape: {input_tensor.shape}")

## Step 4: Extract Features with Pretrained ResNet-18

We load ResNet-18 pretrained on ImageNet. Instead of using the full model for classification,
we strip off the last fully-connected layer and use the 512-dimensional feature vector from
the layer before it. This vector captures visual patterns (texture, edges, shapes) that can
be used downstream for anomaly detection or similarity comparison.

We use `torch.no_grad()` because we're just doing inference — we don't need gradients,
and this makes things faster and uses less memory.

In [ ]:
def load_feature_extractor():
    """
    Loads ResNet-18 pretrained on ImageNet and removes the final
    classification layer so we get raw feature embeddings instead.
    """
    # weights=DEFAULT loads the best available pretrained weights
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Remove the last fully-connected layer (the classifier head)
    # nn.Sequential(*list(...)[:-1]) rebuilds the model without that layer
    feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
    feature_extractor.eval()  # put in eval mode — disables dropout/batchnorm training behavior
    return feature_extractor


def extract_features(model, input_tensor):
    """
    Runs a forward pass through the feature extractor and returns
    a flat 1D numpy array of feature values.
    """
    with torch.no_grad():
        features = model(input_tensor)  # shape: (1, 512, 1, 1)
    # Flatten to a 1D vector of length 512
    return features.squeeze().numpy()


print("Loading pretrained ResNet-18...")
model = load_feature_extractor()

print("Extracting features...")
feature_vector = extract_features(model, input_tensor)

print(f"Feature vector shape : {feature_vector.shape}")
print(f"Feature vector stats : min={feature_vector.min():.4f}, "
      f"max={feature_vector.max():.4f}, mean={feature_vector.mean():.4f}")

## Step 5: Metadata Analysis (EXIF Fraud Signals)

Legitimate drone photos taken at a real property should have:
- **GPS coordinates** embedded in the EXIF data
- **A timestamp** (DateTimeOriginal) showing when the photo was taken

Red flags that could indicate fraud:
- Missing GPS data (photo may not have been taken at the claimed property)
- Missing timestamp (could mean metadata was stripped to hide editing)
- Timestamp much older than the claimed storm date

This function reads those fields and returns a list of any flags found.

In [ ]:
def check_exif_metadata(image_path, claimed_storm_date=None):
    """
    Reads EXIF metadata from a JPEG image and checks for fraud signals.

    Args:
        image_path        : Path to the image file.
        claimed_storm_date: Optional datetime.date — the date the customer
                            claims the storm occurred. Used to sanity-check
                            the photo timestamp.

    Returns:
        A dict with:
          'flags'       : list of fraud signal strings (empty = no flags)
          'gps_present' : bool
          'timestamp'   : str or None
    """
    flags = []
    gps_present = False
    timestamp = None

    img = Image.open(image_path)
    exif_data = img._getexif()  # returns a dict of tag_id -> value, or None

    if exif_data is None:
        flags.append("NO_EXIF_DATA: Image has no EXIF metadata at all.")
        return {"flags": flags, "gps_present": False, "timestamp": None}

    # Build a human-readable dict: tag name → value
    readable_exif = {
        ExifTags.TAGS.get(tag_id, tag_id): value
        for tag_id, value in exif_data.items()
    }

    # --- GPS Check ---
    if "GPSInfo" in readable_exif and readable_exif["GPSInfo"]:
        gps_present = True
    else:
        flags.append("MISSING_GPS: No GPS data found. "
                     "Cannot confirm photo was taken at claimed property.")

    # --- Timestamp Check ---
    raw_ts = readable_exif.get("DateTimeOriginal") or readable_exif.get("DateTime")
    if raw_ts:
        timestamp = raw_ts
        # If a storm date was provided, check if the photo predates it
        if claimed_storm_date is not None:
            try:
                # EXIF timestamp format: "YYYY:MM:DD HH:MM:SS"
                photo_date = datetime.datetime.strptime(raw_ts, "%Y:%m:%d %H:%M:%S").date()
                if photo_date < claimed_storm_date:
                    flags.append(
                        f"TIMESTAMP_PREDATES_STORM: Photo taken on {photo_date}, "
                        f"but storm was claimed on {claimed_storm_date}."
                    )
            except ValueError:
                flags.append(f"TIMESTAMP_PARSE_ERROR: Could not parse timestamp '{raw_ts}'.")
    else:
        flags.append("MISSING_TIMESTAMP: No DateTimeOriginal field found. "
                     "Metadata may have been stripped or image was edited.")

    return {"flags": flags, "gps_present": gps_present, "timestamp": timestamp}


# Run the metadata check on our sample image
# (Placeholder images won't have EXIF, so we expect flags — that's fine for demo purposes)
claimed_date = datetime.date(2024, 5, 15)  # example claimed storm date
metadata_result = check_exif_metadata(SAMPLE_IMAGE_PATH, claimed_storm_date=claimed_date)

print("--- EXIF Metadata Analysis ---")
print(f"GPS Present : {metadata_result['gps_present']}")
print(f"Timestamp   : {metadata_result['timestamp']}")
print(f"Fraud Flags : {len(metadata_result['flags'])} found")
for flag in metadata_result['flags']:
    print(f"  ⚑ {flag}")

## Step 6: Combine into a Fraud Risk Report

This final step aggregates the metadata flags into a simple fraud risk score.
Each flag adds to the risk level. In a production system, the feature vector
from Step 4 would feed into a trained classifier here instead of just being printed.

In [ ]:
def compute_fraud_risk(metadata_result, feature_vector):
    """
    Combines metadata flags and feature vector stats into a fraud risk level.
    
    In a full system, `feature_vector` would be passed to a trained classifier.
    Here we just report it alongside the metadata flags.
    """
    num_flags = len(metadata_result["flags"])

    # Simple rule-based risk level from metadata alone
    if num_flags == 0:
        risk_level = "LOW"
    elif num_flags == 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "HIGH"

    return {
        "risk_level": risk_level,
        "num_flags": num_flags,
        "feature_vector_norm": float(np.linalg.norm(feature_vector)),
    }


risk_report = compute_fraud_risk(metadata_result, feature_vector)

print("=" * 45)
print("  FRAUD RISK REPORT")
print("=" * 45)
print(f"  Risk Level         : {risk_report['risk_level']}")
print(f"  Metadata Flags     : {risk_report['num_flags']}")
print(f"  Feature Vector Norm: {risk_report['feature_vector_norm']:.4f}")
print("  (Feature vector ready for downstream classifier)")
print("=" * 45)

## Next Steps

To extend this into a production-grade pipeline:

1. **Train a classifier** on labeled (fraudulent vs. legitimate) claim images using the ResNet feature vectors.
2. **Expand metadata checks** — cross-reference GPS coordinates against the claimed property address using a geocoding API.
3. **Integrate with severity_scorer.py** — flag claims where the metadata risk is HIGH but the severity score is also very high (suspiciously coincidental).
4. **Add duplicate detection** — hash feature vectors to detect the same image submitted across multiple claims.